# 🎥 Tutorial: Resumen de Videos con Transformers

Este notebook te guía paso a paso en el proceso de resumir videos educativos usando Whisper y modelos Transformer.

## 1. Importar Librerías

In [ ]:
import sys
sys.path.append('../src')

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import whisper
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Transcripción de Audio con Whisper

In [ ]:
# Cargar modelo Whisper
print("Cargando Whisper...")
whisper_model = whisper.load_model("base")
print("Modelo cargado!")

In [ ]:
# Transcribir un archivo de audio
# audio_path = "../videos/mi_audio.mp3"
# result = whisper_model.transcribe(audio_path, language="es")
# texto_transcrito = result["text"]

# Para este ejemplo, usaremos un texto de muestra
texto_transcrito = """
En este video hablaremos sobre las redes neuronales artificiales y su importancia en el aprendizaje profundo.
Las redes neuronales son modelos computacionales inspirados en el funcionamiento del cerebro humano.
Están compuestas por capas de neuronas artificiales que procesan información de manera jerárquica.
La capa de entrada recibe los datos, las capas ocultas realizan transformaciones complejas,
y la capa de salida produce el resultado final. Durante el entrenamiento, la red ajusta sus pesos
mediante el algoritmo de retropropagación para minimizar el error. Las redes neuronales profundas
han revolucionado campos como la visión por computadora, el procesamiento de lenguaje natural
y el reconocimiento de voz, logrando resultados sorprendentes en tareas que antes eran imposibles
para las computadoras.
"""

print("Texto transcrito:")
print(texto_transcrito)
print(f"\nLongitud: {len(texto_transcrito)} caracteres")

## 3. Cargar Modelo de Resumen Pre-entrenado

In [ ]:
# Cargar modelo mT5 para español (o T5 para inglés)
model_name = "google/mt5-small"
print(f"Cargando modelo {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"Modelo cargado en {device}!")

## 4. Generar Resumen (Sin Fine-tuning)

In [ ]:
def generar_resumen(texto, max_length=128, min_length=30):
    """Genera un resumen del texto dado"""
    # Agregar prefix para T5/mT5
    input_text = "resumir: " + texto
    
    # Tokenizar
    inputs = tokenizer(
        input_text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Generar resumen
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )
    
    # Decodificar
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Generar resumen
resumen = generar_resumen(texto_transcrito)

print("="*60)
print("RESUMEN GENERADO")
print("="*60)
print(resumen)
print(f"\nLongitud del resumen: {len(resumen)} caracteres")
print(f"Tasa de compresión: {len(resumen)/len(texto_transcrito):.2%}")

## 5. Preparar Dataset para Fine-tuning

In [ ]:
from datasets import load_dataset

# Cargar dataset público para entrenamiento
print("Cargando dataset XSum...")
dataset = load_dataset("xsum", split="train[:1000]")  # Solo 1000 ejemplos para demo

print(f"Dataset cargado: {len(dataset)} ejemplos")
print("\nEjemplo:")
print(f"Documento: {dataset[0]['document'][:200]}...")
print(f"Resumen: {dataset[0]['summary']}")

## 6. Preprocesar Dataset

In [ ]:
def preprocess_function(examples):
    """Preprocesa ejemplos para T5"""
    inputs = ["resumir: " + doc for doc in examples["document"]]
    
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["summary"],
            max_length=128,
            truncation=True,
            padding="max_length"
        )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Procesar dataset
print("Preprocesando dataset...")
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset.column_names
)

print("Dataset preprocesado!")
print(f"Ejemplo tokenizado: {tokenized_dataset[0].keys()}")

## 7. Fine-tuning (Opcional - Requiere tiempo y recursos)

**NOTA**: Este paso puede tomar mucho tiempo. Para pruebas, usa `max_steps=100` en lugar de `num_train_epochs`.

In [ ]:
# COMENTADO: Descomentar solo si quieres entrenar
"""
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# Configurar argumentos
training_args = Seq2SeqTrainingArguments(
    output_dir="../models/mt5-finetuned",
    evaluation_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    max_steps=100,  # Solo 100 pasos para demo
    save_steps=50,
    logging_steps=10,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Crear trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Entrenar
print("Iniciando entrenamiento...")
trainer.train()
print("Entrenamiento completado!")

# Guardar modelo
trainer.save_model("../models/mt5-finetuned")
tokenizer.save_pretrained("../models/mt5-finetuned")
"""

print("Para entrenar, descomenta el código en esta celda")

## 8. Comparar Modelos

Compara el modelo base vs. el modelo con fine-tuning

In [ ]:
# Textos de prueba
textos_prueba = [
    texto_transcrito,
    """El machine learning es una rama de la inteligencia artificial que permite a las computadoras 
    aprender de los datos sin ser programadas explícitamente. Utiliza algoritmos que mejoran 
    automáticamente con la experiencia.""",
]

# Generar resúmenes
print("Generando resúmenes...\n")
for i, texto in enumerate(textos_prueba, 1):
    resumen = generar_resumen(texto)
    print(f"Texto {i}:")
    print(f"Original: {texto[:100]}...")
    print(f"Resumen: {resumen}")
    print(f"Compresión: {len(resumen)/len(texto):.2%}\n")
    print("-"*60)

## 9. Métricas de Evaluación (ROUGE)

In [ ]:
import evaluate

# Cargar métrica ROUGE
rouge = evaluate.load("rouge")

# Ejemplo de evaluación
prediccion = ["Las redes neuronales son modelos inspirados en el cerebro."]
referencia = ["Las redes neuronales artificiales imitan el funcionamiento cerebral."]

results = rouge.compute(predictions=prediccion, references=referencia)

print("Métricas ROUGE:")
for key, value in results.items():
    print(f"  {key}: {value:.4f}")

## 10. Conclusiones

En este notebook has aprendido:
- ✅ Cómo usar Whisper para transcribir audio
- ✅ Cómo cargar modelos T5/mT5 para resumen
- ✅ Cómo preparar datos para fine-tuning
- ✅ Cómo generar resúmenes automáticos
- ✅ Cómo evaluar con métricas ROUGE

**Próximos pasos:**
1. Entrenar con tu propio dataset
2. Optimizar hiperparámetros
3. Probar diferentes modelos (BART, PEGASUS)
4. Integrar en una aplicación web